# Pipeline statistique - transmission EURO STOXX 50 future -> courbe des futures sur dividendes

Ce notebook est conçu pour 7 datasets :

- `F` : future EURO STOXX 50
- `D0`, `D1`, `D2`, `D3`, `D4`, `D5` : futures sur dividendes de maturité 0y à 5y

Chaque dataset contient :

- `ts` : timestamp
- `mid` : mid-price
- `V` : volume
- `Imbalance` : imbalance d'ordre-flow ou de volume

Hypothèse de travail : les données ont été extraites intraday entre 9h et 17h30, agrégées à 1 seconde.

Objectif final : tester si l'information de l'index future se transmet à la courbe des dividendes, puis vérifier si cette transmission est exploitable pour le market making.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

# Optionnels, utiles si installés
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False

try:
    from sklearn.linear_model import ElasticNetCV, RidgeCV, LassoCV
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import r2_score, mean_absolute_error
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False


## 1. Paramètres

Adapte les chemins et paramètres. Le notebook suppose un dictionnaire de DataFrames :

```python
data = {
    'F': df_fesx,
    'D0': df_d0,
    ...,
    'D5': df_d5,
}
```

Chaque DataFrame doit avoir les colonnes `ts`, `mid`, `V`, `Imbalance`.


In [ ]:
# =========================
# PARAMÈTRES UTILISATEUR
# =========================

TS_COL = 'ts'
MID_COL = 'mid'
VOL_COL = 'V'
IMB_COL = 'Imbalance'

INSTRUMENTS = ['F', 'D0', 'D1', 'D2', 'D3', 'D4', 'D5']
DIVS = ['D0', 'D1', 'D2', 'D3', 'D4', 'D5']

SESSION_START = '09:00:00'
SESSION_END = '17:30:00'
BASE_FREQ = '1s'
MODEL_FREQ = '10min'
DAILY_FREQ = '1D'

# Seuil de stale price : à adapter selon liquidité.
STALE_THRESHOLD = pd.Timedelta('10min')

# Nombre de lags sur la grille 10min. 6 lags = 1 heure.
N_LAGS = 6


## 2. Chargement des données

Décommente et adapte cette section à tes fichiers. Le notebook ne suppose pas un format particulier.


In [ ]:
# Exemple CSV
# df_f = pd.read_csv('F.csv')
# df_d0 = pd.read_csv('D0.csv')
# ...
# data = {'F': df_f, 'D0': df_d0, 'D1': df_d1, 'D2': df_d2, 'D3': df_d3, 'D4': df_d4, 'D5': df_d5}

# Exemple Parquet
# data = {name: pd.read_parquet(f'{name}.parquet') for name in INSTRUMENTS}

# Placeholder : décommenter après chargement
# data.keys()


## 3. Fonctions d'audit et de nettoyage

Règles méthodologiques :

- `V` manquant peut être remplacé par 0 seulement si le NA vient d'un resampling sans activité.
- `Imbalance` manquant peut être remplacé par 0 seulement si `V=0`.
- `mid` brut doit être conservé avec ses NA pour l'audit.
- Pour l'analyse, on construit `mid_ffill` et un `stale_flag`.


In [ ]:
def robust_zscore(x):
    x = pd.Series(x).astype(float)
    med = x.median()
    mad = (x - med).abs().median()
    if mad == 0 or np.isnan(mad):
        return pd.Series(np.nan, index=x.index)
    return 0.6745 * (x - med) / mad


def prepare_single_dataset(df, name, ts_col=TS_COL, mid_col=MID_COL, vol_col=VOL_COL, imb_col=IMB_COL,
                           session_start=SESSION_START, session_end=SESSION_END):
    out = df.copy()

    # Timestamp conversion
    if not np.issubdtype(out[ts_col].dtype, np.datetime64):
        if np.issubdtype(out[ts_col].dtype, np.number):
            med = out[ts_col].dropna().median()
            if med > 1e12:
                out[ts_col] = pd.to_datetime(out[ts_col], unit='ms')
            elif med > 1e9:
                out[ts_col] = pd.to_datetime(out[ts_col], unit='s')
            else:
                out[ts_col] = pd.Timestamp('2000-01-01') + pd.to_timedelta(out[ts_col], unit='s')
        else:
            out[ts_col] = pd.to_datetime(out[ts_col])

    out = out.sort_values(ts_col).set_index(ts_col)

    # Intraday filter
    out = out.between_time(session_start, session_end)

    # Numeric conversion
    for c in [mid_col, vol_col, imb_col]:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    # Keep raw flags
    out['mid_raw_missing'] = out[mid_col].isna()
    out['V_raw_missing'] = out[vol_col].isna()
    out['Imb_raw_missing'] = out[imb_col].isna()

    # Volume: fill missing with 0 only as convention for no activity.
    out[vol_col] = out[vol_col].fillna(0.0)
    out.loc[out[vol_col] < 0, vol_col] = np.nan

    # Imbalance: fill NA with 0 only when V == 0. Keep NA when V > 0.
    mask_imb_na_zero_vol = out[imb_col].isna() & (out[vol_col].fillna(0) == 0)
    out.loc[mask_imb_na_zero_vol, imb_col] = 0.0

    # Validity flags
    out['invalid_mid'] = out[mid_col].notna() & (out[mid_col] <= 0)
    out['invalid_V'] = out[vol_col].isna() | (out[vol_col] < 0)
    out['invalid_Imb'] = out[imb_col].notna() & (out[imb_col].abs() > 1)
    out['active'] = out[vol_col].fillna(0) > 0

    # Remove clearly invalid mid from analytic columns, keep raw information.
    out.loc[out['invalid_mid'], mid_col] = np.nan

    # Rename columns with instrument suffix
    out = out.rename(columns={mid_col: f'mid_{name}', vol_col: f'V_{name}', imb_col: f'Imb_{name}'})

    keep_cols = [f'mid_{name}', f'V_{name}', f'Imb_{name}', 'active', 'mid_raw_missing', 'V_raw_missing', 'Imb_raw_missing', 'invalid_mid', 'invalid_V', 'invalid_Imb']
    out = out[keep_cols]
    out = out.rename(columns={
        'active': f'active_{name}',
        'mid_raw_missing': f'mid_raw_missing_{name}',
        'V_raw_missing': f'V_raw_missing_{name}',
        'Imb_raw_missing': f'Imb_raw_missing_{name}',
        'invalid_mid': f'invalid_mid_{name}',
        'invalid_V': f'invalid_V_{name}',
        'invalid_Imb': f'invalid_Imb_{name}',
    })

    return out


def audit_single_prepared(df_prep, name):
    mid = df_prep[f'mid_{name}']
    V = df_prep[f'V_{name}']
    Imb = df_prep[f'Imb_{name}']
    rows = {
        'instrument': name,
        'n_obs': len(df_prep),
        'start': df_prep.index.min(),
        'end': df_prep.index.max(),
        'n_days': pd.Series(df_prep.index.date).nunique(),
        'mid_missing_pct': mid.isna().mean(),
        'V_missing_pct_after_clean': V.isna().mean(),
        'Imb_missing_pct_after_clean': Imb.isna().mean(),
        'active_ratio': (V.fillna(0) > 0).mean(),
        'invalid_mid_count': df_prep[f'invalid_mid_{name}'].sum(),
        'invalid_V_count': df_prep[f'invalid_V_{name}'].sum(),
        'invalid_Imb_count': df_prep[f'invalid_Imb_{name}'].sum(),
        'mid_unique_ratio': mid.nunique(dropna=True) / max(1, mid.notna().sum()),
    }
    return rows


## 4. Préparer les 7 datasets et construire le panel 1 seconde

Cette étape produit un panel large avec toutes les variables instrument par instrument.


In [ ]:
def build_base_panel(data, base_freq=BASE_FREQ):
    prepared = {}
    audits = []

    for name, df in data.items():
        p = prepare_single_dataset(df, name)
        prepared[name] = p
        audits.append(audit_single_prepared(p, name))

    audit_df = pd.DataFrame(audits)

    # Outer join on timestamps, then resample to 1s grid
    panel = None
    for name in INSTRUMENTS:
        dfp = prepared[name]
        if panel is None:
            panel = dfp
        else:
            panel = panel.join(dfp, how='outer')

    panel = panel.sort_index()

    # Resample to base 1s grid
    agg = {}
    for name in INSTRUMENTS:
        agg[f'mid_{name}'] = 'last'
        agg[f'V_{name}'] = 'sum'
        # For imbalance, volume-weighted aggregation is handled below manually.
        agg[f'Imb_{name}'] = 'last'
        for flag in [f'active_{name}', f'mid_raw_missing_{name}', f'V_raw_missing_{name}', f'Imb_raw_missing_{name}', f'invalid_mid_{name}', f'invalid_V_{name}', f'invalid_Imb_{name}']:
            if flag in panel.columns:
                agg[flag] = 'max'

    panel_1s = panel.resample(base_freq).agg(agg)

    # Fill volumes and active flags after resampling
    for name in INSTRUMENTS:
        panel_1s[f'V_{name}'] = panel_1s[f'V_{name}'].fillna(0.0)
        panel_1s[f'active_{name}'] = panel_1s[f'active_{name}'].fillna(False).astype(bool)
        # Imbalance convention: NA -> 0 if V=0; otherwise keep NA.
        imb = f'Imb_{name}'
        V = f'V_{name}'
        panel_1s.loc[panel_1s[imb].isna() & (panel_1s[V] == 0), imb] = 0.0

    return panel_1s, audit_df, prepared

# Utilisation après chargement :
# panel_1s, audit_df, prepared = build_base_panel(data)
# audit_df
# panel_1s.head()


## 5. Construire les mids analytiques, stale_age et stale_flag

On conserve le mid brut, mais on crée un `mid_ffill` pour calculer les variations. Chaque `mid_ffill` est accompagné d'un `stale_age` et d'un `stale_flag`.


In [ ]:
def add_mid_ffill_and_stale(panel, stale_threshold=STALE_THRESHOLD):
    out = panel.copy()

    for name in INSTRUMENTS:
        raw = out[f'mid_{name}']
        ffilled = raw.ffill()
        out[f'mid_ffill_{name}'] = ffilled

        # Last valid observation timestamp
        valid_ts = pd.Series(out.index.where(raw.notna()), index=out.index).ffill()
        age = out.index.to_series() - valid_ts
        out[f'stale_age_{name}'] = age
        out[f'stale_{name}'] = age > stale_threshold

        # Returns/changes on 1s grid
        if name == 'F':
            out[f'r_{name}'] = np.log(out[f'mid_ffill_{name}']).diff()
        else:
            out[f'delta_{name}'] = out[f'mid_ffill_{name}'].diff()
            out[f'r_{name}'] = np.log(out[f'mid_ffill_{name}']).diff()

    return out

# panel_1s = add_mid_ffill_and_stale(panel_1s)


## 6. Audit post-panel

Tableau par instrument : missing, active ratio, stale ratio, variation non nulle, volume.


In [ ]:
def audit_panel(panel):
    rows = []
    for name in INSTRUMENTS:
        mid = panel[f'mid_{name}']
        mid_ff = panel.get(f'mid_ffill_{name}', mid.ffill())
        V = panel[f'V_{name}']
        Imb = panel[f'Imb_{name}']
        delta = mid_ff.diff()
        rows.append({
            'instrument': name,
            'n_obs': len(panel),
            'mid_raw_missing_pct': mid.isna().mean(),
            'active_ratio': (V > 0).mean(),
            'stale_ratio': panel.get(f'stale_{name}', pd.Series(False, index=panel.index)).mean(),
            'zero_volume_ratio': (V == 0).mean(),
            'imb_missing_pct': Imb.isna().mean(),
            'nonzero_mid_change_ratio': (delta.fillna(0) != 0).mean(),
            'mean_volume': V.mean(),
            'median_volume': V.median(),
            'mean_abs_imb_when_active': Imb[V>0].abs().mean(),
        })
    return pd.DataFrame(rows)

# panel_audit = audit_panel(panel_1s)
# panel_audit


## 7. Agrégation générique

On agrège depuis la base 1s vers 1min, 10min ou daily.

Pour une imbalance déjà normalisée, on utilise une moyenne pondérée par volume :

$$
Imb_{bucket} = \frac{\sum_t Imb_t V_t}{\sum_t V_t}
$$

si le volume du bucket est positif ; sinon 0.


In [ ]:
def aggregate_panel(panel, freq='10min'):
    pieces = []
    idx = panel.resample(freq).size().index
    out = pd.DataFrame(index=idx)

    for name in INSTRUMENTS:
        mid_col = f'mid_ffill_{name}' if f'mid_ffill_{name}' in panel.columns else f'mid_{name}'
        V_col = f'V_{name}'
        Imb_col = f'Imb_{name}'

        out[f'mid_{name}'] = panel[mid_col].resample(freq).last()
        out[f'V_{name}'] = panel[V_col].resample(freq).sum().fillna(0.0)

        weighted_imb_num = (panel[Imb_col].fillna(0) * panel[V_col].fillna(0)).resample(freq).sum()
        vol_sum = panel[V_col].fillna(0).resample(freq).sum()
        out[f'Imb_{name}'] = np.where(vol_sum > 0, weighted_imb_num / vol_sum, 0.0)
        out[f'active_{name}'] = (vol_sum > 0)

        if f'stale_{name}' in panel.columns:
            out[f'stale_ratio_{name}'] = panel[f'stale_{name}'].resample(freq).mean()
            out[f'stale_{name}'] = out[f'stale_ratio_{name}'] > 0.5

        if name == 'F':
            out[f'r_{name}'] = np.log(out[f'mid_{name}']).diff()
        else:
            out[f'delta_{name}'] = out[f'mid_{name}'].diff()
            out[f'r_{name}'] = np.log(out[f'mid_{name}']).diff()

    # Time features
    out['date'] = pd.to_datetime(out.index.date)
    out['hour'] = out.index.hour
    out['minute'] = out.index.minute
    out['tod'] = out.index.hour + out.index.minute / 60.0
    out['year'] = out.index.year

    return out

# panel_10m = aggregate_panel(panel_1s, MODEL_FREQ)
# panel_1m = aggregate_panel(panel_1s, '1min')
# panel_daily = aggregate_panel(panel_1s, '1D')


## 8. Analyse descriptive par maturité

In [ ]:
def descriptive_by_maturity(panel_agg):
    rows = []
    for name in DIVS:
        rows.append({
            'maturity': name,
            'n_obs': len(panel_agg),
            'active_ratio': panel_agg[f'active_{name}'].mean(),
            'stale_ratio': panel_agg.get(f'stale_{name}', pd.Series(False, index=panel_agg.index)).mean(),
            'mean_V': panel_agg[f'V_{name}'].mean(),
            'median_V': panel_agg[f'V_{name}'].median(),
            'mean_abs_Imb_active': panel_agg.loc[panel_agg[f'active_{name}'], f'Imb_{name}'].abs().mean(),
            'std_delta': panel_agg[f'delta_{name}'].std(),
            'nonzero_delta_ratio': (panel_agg[f'delta_{name}'].fillna(0) != 0).mean(),
        })
    return pd.DataFrame(rows)

# desc_10m = descriptive_by_maturity(panel_10m)
# desc_10m


In [ ]:
def plot_descriptive_by_maturity(desc):
    for col in ['active_ratio', 'stale_ratio', 'mean_V', 'std_delta', 'nonzero_delta_ratio']:
        if col in desc.columns:
            plt.figure(figsize=(8,4))
            plt.bar(desc['maturity'], desc[col])
            plt.title(col + ' by maturity')
            plt.grid(True, alpha=0.3)
            plt.show()

# plot_descriptive_by_maturity(desc_10m)


## 9. Détection des valeurs extrêmes et aberrantes

On flagge les variations extrêmes par maturité via z-score robuste. Il ne faut pas supprimer automatiquement : on garde un flag.


In [ ]:
def add_outlier_flags(panel_agg, z_threshold=8.0):
    out = panel_agg.copy()
    for name in INSTRUMENTS:
        if name == 'F':
            x = out[f'r_{name}']
        else:
            x = out[f'delta_{name}']
        z = robust_zscore(x)
        out[f'robust_z_{name}'] = z
        out[f'outlier_{name}'] = z.abs() > z_threshold

        # Volume extreme flag
        V = out[f'V_{name}']
        q = V.quantile(0.995)
        out[f'extreme_volume_{name}'] = V > q
    return out

# panel_10m = add_outlier_flags(panel_10m)


## 10. Analyse de la courbe des dividendes

On construit Level, Slope, Curvature et éventuellement PCA.


In [ ]:
def add_curve_factors(panel_agg):
    out = panel_agg.copy()
    mids = pd.concat([out[f'mid_{m}'] for m in DIVS], axis=1)
    mids.columns = DIVS

    out['level_D'] = mids.mean(axis=1)
    out['slope_D_5_0'] = out['mid_D5'] - out['mid_D0']
    # Curvature simple avec D0, D2, D5
    out['curvature_D'] = out['mid_D0'] - 2*out['mid_D2'] + out['mid_D5']

    out['d_level_D'] = out['level_D'].diff()
    out['d_slope_D_5_0'] = out['slope_D_5_0'].diff()
    out['d_curvature_D'] = out['curvature_D'].diff()
    return out

# panel_10m = add_curve_factors(panel_10m)
# panel_daily = add_curve_factors(panel_daily)


In [ ]:
def plot_curve_factors(panel_agg):
    for col in ['level_D', 'slope_D_5_0', 'curvature_D']:
        if col in panel_agg.columns:
            plt.figure(figsize=(12,4))
            plt.plot(panel_agg.index, panel_agg[col])
            plt.title(col)
            plt.grid(True, alpha=0.3)
            plt.show()

# plot_curve_factors(panel_10m)


## 11. Corrélations contemporaines et laggées

Les heatmaps maturité x lag sont centrales pour répondre à la question de transmission.


In [ ]:
def lagged_corr_heatmap(panel_agg, predictor_col, target_prefix='delta_', maturities=DIVS, max_lag=N_LAGS):
    rows = []
    for m in maturities:
        y = panel_agg[f'{target_prefix}{m}']
        for lag in range(max_lag + 1):
            x = panel_agg[predictor_col].shift(lag)
            tmp = pd.concat([x, y], axis=1).dropna()
            corr = tmp.iloc[:,0].corr(tmp.iloc[:,1]) if len(tmp) > 5 else np.nan
            rows.append({'maturity': m, 'lag': lag, 'corr': corr})
    heat = pd.DataFrame(rows).pivot(index='maturity', columns='lag', values='corr')
    return heat


def plot_heatmap(df, title):
    plt.figure(figsize=(10,4))
    plt.imshow(df.values, aspect='auto')
    plt.colorbar(label='corr')
    plt.xticks(range(df.shape[1]), df.columns)
    plt.yticks(range(df.shape[0]), df.index)
    plt.title(title)
    plt.xlabel('Lag')
    plt.ylabel('Maturity')
    plt.show()

# heat_rF = lagged_corr_heatmap(panel_10m, 'r_F')
# plot_heatmap(heat_rF, 'Corr laggée: r_F -> DeltaD')

# heat_imbF = lagged_corr_heatmap(panel_10m, 'Imb_F')
# plot_heatmap(heat_imbF, 'Corr laggée: Imb_F -> DeltaD')

# heat_volF = lagged_corr_heatmap(panel_10m, 'V_F')
# plot_heatmap(heat_volF, 'Corr laggée: V_F -> DeltaD')


## 12. Construction du panel long pour régression

On transforme le panel large en panel long : une ligne par timestamp et maturité.


In [ ]:
def make_long_panel(panel_agg, n_lags=N_LAGS):
    base = panel_agg.copy()

    # Lagged index features
    for lag in range(n_lags + 1):
        base[f'r_F_lag{lag}'] = base['r_F'].shift(lag)
        base[f'Imb_F_lag{lag}'] = base['Imb_F'].shift(lag)
        base[f'logV_F_lag{lag}'] = np.log1p(base['V_F']).shift(lag)

    rows = []
    for m in DIVS:
        cols = {
            'ts': base.index,
            'maturity': m,
            'y_deltaD': base[f'delta_{m}'].values,
            'mid_D': base[f'mid_{m}'].values,
            'V_D': base[f'V_{m}'].values,
            'logV_D': np.log1p(base[f'V_{m}']).values,
            'Imb_D': base[f'Imb_{m}'].values,
            'active_D': base[f'active_{m}'].values,
            'stale_D': base.get(f'stale_{m}', pd.Series(False, index=base.index)).values,
            'hour': base['hour'].values,
            'tod': base['tod'].values,
            'date': base['date'].values,
            'year': base['year'].values,
        }
        for lag in range(n_lags + 1):
            cols[f'r_F_lag{lag}'] = base[f'r_F_lag{lag}'].values
            cols[f'Imb_F_lag{lag}'] = base[f'Imb_F_lag{lag}'].values
            cols[f'logV_F_lag{lag}'] = base[f'logV_F_lag{lag}'].values
        rows.append(pd.DataFrame(cols))

    long = pd.concat(rows, axis=0, ignore_index=True)
    return long

# long_10m = make_long_panel(panel_10m)
# long_10m.head()


## 13. Régressions par maturité

Modèle central par maturité :

$$
\Delta D_t^m = \alpha_m + \sum_l \beta_{m,l} r^F_{t-l} + \sum_l \gamma_{m,l} Imb^F_{t-l} + \sum_l \eta_{m,l}\log(1+V^F_{t-l}) + \epsilon_t^m
$$


In [ ]:
def fit_ols_by_maturity(long_df, n_lags=N_LAGS, use_active_only=False, drop_stale=True):
    if not STATSMODELS_AVAILABLE:
        raise ImportError('statsmodels is required for OLS regression.')

    results = {}
    coef_rows = []

    predictors = []
    for lag in range(n_lags + 1):
        predictors += [f'r_F_lag{lag}', f'Imb_F_lag{lag}', f'logV_F_lag{lag}']
    predictors += ['Imb_D', 'logV_D']

    for m in DIVS:
        dfm = long_df[long_df['maturity'] == m].copy()
        if use_active_only:
            dfm = dfm[dfm['active_D']]
        if drop_stale:
            dfm = dfm[~dfm['stale_D']]
        dfm = dfm[['y_deltaD'] + predictors].replace([np.inf, -np.inf], np.nan).dropna()

        if len(dfm) < 50:
            print(f'Not enough observations for {m}: {len(dfm)}')
            continue

        X = sm.add_constant(dfm[predictors])
        y = dfm['y_deltaD']
        # HAC robust standard errors for autocorrelation
        model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': max(1, n_lags)})
        results[m] = model
        for var, coef in model.params.items():
            coef_rows.append({'maturity': m, 'variable': var, 'coef': coef, 't': model.tvalues[var], 'pvalue': model.pvalues[var], 'nobs': int(model.nobs), 'rsq': model.rsquared})

    return results, pd.DataFrame(coef_rows)

# ols_results, coef_df = fit_ols_by_maturity(long_10m)
# coef_df.head()


In [ ]:
def coef_heatmap(coef_df, prefix='r_F_lag'):
    tmp = coef_df[coef_df['variable'].str.startswith(prefix)].copy()
    tmp['lag'] = tmp['variable'].str.replace(prefix, '', regex=False).astype(int)
    heat = tmp.pivot(index='maturity', columns='lag', values='coef')
    return heat

# beta_heat = coef_heatmap(coef_df, 'r_F_lag')
# plot_heatmap(beta_heat, 'Coefficients beta_{m,l}: r_F laggé -> DeltaD')

# gamma_heat = coef_heatmap(coef_df, 'Imb_F_lag')
# plot_heatmap(gamma_heat, 'Coefficients gamma_{m,l}: Imb_F laggé -> DeltaD')


## 14. Régression panel avec effets fixes maturité et heure

La formule ci-dessous garde les maturités séparées via effets fixes et interactions. Pour un premier panel, on peut inclure les lags principaux et `C(maturity)`, `C(hour)`.


In [ ]:
def fit_panel_formula(long_df, n_lags=N_LAGS, drop_stale=True):
    if not STATSMODELS_AVAILABLE:
        raise ImportError('statsmodels is required.')

    df = long_df.copy()
    if drop_stale:
        df = df[~df['stale_D']]

    # Variables candidates
    terms = []
    for lag in range(n_lags + 1):
        # Interactions with maturity to estimate beta/gamma per maturity
        terms.append(f'C(maturity):r_F_lag{lag}')
        terms.append(f'C(maturity):Imb_F_lag{lag}')
        terms.append(f'C(maturity):logV_F_lag{lag}')

    terms += ['Imb_D', 'logV_D', 'C(maturity)', 'C(hour)']
    formula = 'y_deltaD ~ ' + ' + '.join(terms)

    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['y_deltaD'] + [c for c in df.columns if c.startswith(('r_F_lag','Imb_F_lag','logV_F_lag'))] + ['Imb_D','logV_D','maturity','hour'])

    model = smf.ols(formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['date']})
    return model

# panel_model = fit_panel_formula(long_10m)
# print(panel_model.summary())


## 15. Modèles de courbe : Level, Slope, Curvature

In [ ]:
def fit_curve_models(panel_agg, n_lags=N_LAGS):
    if not STATSMODELS_AVAILABLE:
        raise ImportError('statsmodels required.')
    df = panel_agg.copy()

    for lag in range(n_lags + 1):
        df[f'r_F_lag{lag}'] = df['r_F'].shift(lag)
        df[f'Imb_F_lag{lag}'] = df['Imb_F'].shift(lag)
        df[f'logV_F_lag{lag}'] = np.log1p(df['V_F']).shift(lag)

    predictors = []
    for lag in range(n_lags + 1):
        predictors += [f'r_F_lag{lag}', f'Imb_F_lag{lag}', f'logV_F_lag{lag}']

    results = {}
    for ycol in ['d_level_D', 'd_slope_D_5_0', 'd_curvature_D']:
        if ycol not in df.columns:
            continue
        tmp = df[[ycol] + predictors].replace([np.inf, -np.inf], np.nan).dropna()
        X = sm.add_constant(tmp[predictors])
        y = tmp[ycol]
        results[ycol] = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': max(1, n_lags)})
    return results

# curve_results = fit_curve_models(panel_10m)
# for k,v in curve_results.items():
#     print('\n', k)
#     print(v.summary())


## 16. Sélection statistique avec Elastic Net en walk-forward

Cette section donne une base de sélection de variables sans fuite temporelle. On split par dates.


In [ ]:
def temporal_train_test_split(df, train_end, test_start=None):
    train_end = pd.to_datetime(train_end)
    if test_start is None:
        test_start = train_end
    else:
        test_start = pd.to_datetime(test_start)
    train = df[pd.to_datetime(df['date']) <= train_end].copy()
    test = df[pd.to_datetime(df['date']) > test_start].copy()
    return train, test


def fit_elastic_net_by_maturity(long_df, maturity, train_end='2025-12-31', n_lags=N_LAGS, drop_stale=True):
    if not SKLEARN_AVAILABLE:
        raise ImportError('scikit-learn required.')

    df = long_df[long_df['maturity'] == maturity].copy()
    if drop_stale:
        df = df[~df['stale_D']]

    features = []
    for lag in range(n_lags + 1):
        features += [f'r_F_lag{lag}', f'Imb_F_lag{lag}', f'logV_F_lag{lag}']
    features += ['Imb_D', 'logV_D', 'hour']

    df = df[['date','y_deltaD'] + features].replace([np.inf, -np.inf], np.nan).dropna()
    train, test = temporal_train_test_split(df, train_end=train_end)

    X_train, y_train = train[features], train['y_deltaD']
    X_test, y_test = test[features], test['y_deltaD']

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('enet', ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], alphas=None, cv=5, max_iter=10000))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test) if len(test) else np.array([])

    metrics = {}
    if len(test):
        benchmark = np.repeat(y_train.mean(), len(y_test))
        metrics['r2_oos_vs_mean'] = 1 - np.sum((y_test - pred)**2) / np.sum((y_test - benchmark)**2)
        metrics['mae'] = mean_absolute_error(y_test, pred)
        metrics['hit_ratio'] = np.mean(np.sign(pred) == np.sign(y_test))
        metrics['corr_pred_realized'] = np.corrcoef(pred, y_test)[0,1] if len(y_test)>1 else np.nan
    return model, metrics, pd.DataFrame({'y': y_test, 'pred': pred}, index=test.index)

# enet_model_D0, metrics_D0, pred_D0 = fit_elastic_net_by_maturity(long_10m, 'D0', train_end='2025-06-30')
# metrics_D0


## 17. Validation hors-échantillon et backtest proxy après coûts

Un signal statistique est exploitable seulement si son edge dépasse les coûts.


In [ ]:
def backtest_proxy(pred_df, cost=0.0):
    """
    pred_df must contain columns y and pred.
    Position = +1 if pred > cost, -1 if pred < -cost, else 0.
    PnL = position * y - cost * abs(position)
    """
    out = pred_df.dropna().copy()
    out['position'] = np.where(out['pred'] > cost, 1, np.where(out['pred'] < -cost, -1, 0))
    out['pnl'] = out['position'] * out['y'] - cost * np.abs(out['position'])
    summary = {
        'n_obs': len(out),
        'n_trades': int((out['position'] != 0).sum()),
        'trade_ratio': (out['position'] != 0).mean(),
        'mean_pnl': out['pnl'].mean(),
        'sum_pnl': out['pnl'].sum(),
        'hit_ratio_trades': np.mean(np.sign(out.loc[out['position']!=0, 'position']) == np.sign(out.loc[out['position']!=0, 'y'])) if (out['position'] != 0).sum() else np.nan,
    }
    return out, summary

# bt_D0, bt_summary_D0 = backtest_proxy(pred_D0, cost=0.0)
# bt_summary_D0


## 18. Rapport automatique minimal

Cette fonction assemble les principales sorties à inspecter.


In [ ]:
def run_core_pipeline(data):
    panel_1s, audit_df, prepared = build_base_panel(data)
    panel_1s = add_mid_ffill_and_stale(panel_1s)
    panel_audit = audit_panel(panel_1s)

    panel_10m = aggregate_panel(panel_1s, MODEL_FREQ)
    panel_10m = add_outlier_flags(panel_10m)
    panel_10m = add_curve_factors(panel_10m)

    panel_daily = aggregate_panel(panel_1s, '1D')
    panel_daily = add_curve_factors(panel_daily)

    desc_10m = descriptive_by_maturity(panel_10m)
    heat_rF = lagged_corr_heatmap(panel_10m, 'r_F')
    heat_imbF = lagged_corr_heatmap(panel_10m, 'Imb_F')
    heat_volF = lagged_corr_heatmap(panel_10m, 'V_F')
    long_10m = make_long_panel(panel_10m)

    return {
        'audit_raw': audit_df,
        'panel_1s': panel_1s,
        'panel_audit': panel_audit,
        'panel_10m': panel_10m,
        'panel_daily': panel_daily,
        'desc_10m': desc_10m,
        'heat_rF': heat_rF,
        'heat_imbF': heat_imbF,
        'heat_volF': heat_volF,
        'long_10m': long_10m,
    }

# result = run_core_pipeline(data)
# result['panel_audit']
# result['desc_10m']
# plot_heatmap(result['heat_rF'], 'Lagged correlation r_F -> DeltaD')
# plot_heatmap(result['heat_imbF'], 'Lagged correlation Imb_F -> DeltaD')


## 19. Ordre recommandé d'utilisation

1. Charger les 7 datasets dans `data`.
2. Exécuter `run_core_pipeline(data)`.
3. Inspecter `audit_raw` et `panel_audit`.
4. Vérifier `desc_10m`, surtout `active_ratio`, `stale_ratio`, `nonzero_delta_ratio`.
5. Regarder les heatmaps de corrélations laggées.
6. Estimer `fit_ols_by_maturity`.
7. Estimer `fit_panel_formula`.
8. Tester `fit_elastic_net_by_maturity` en walk-forward.
9. Tester le backtest proxy avec un coût réaliste.

Conclusion à rechercher :

- Quels lags de `r_F`, `Imb_F`, `logV_F` prédisent `DeltaD_m` ?
- Cette relation est-elle stable par maturité ?
- Est-elle robuste hors-échantillon ?
- Le signal dépasse-t-il les coûts de trading ?
